In [3]:
from pathlib import Path
import numpy as np
from typing import Any
import sys

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Specify the cache directory here
CACHE_DIR = "/zfsauton/scratch/mineuih/waymax_rs/qa_cache/"  # Change this path as needed

def _split_cache_arrays(npz_data: np.lib.npyio.NpzFile) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray], dict[str, np.ndarray]]:
    """Split cache arrays into features, aux, and metadata."""
    features: dict[str, np.ndarray] = {}
    aux: dict[str, np.ndarray] = {}
    metadata: dict[str, np.ndarray] = {}
    for key in npz_data.files:
        if key.startswith("features/"):
            features[key.split("/", 1)[1]] = npz_data[key]
        elif key.startswith("aux/"):
            aux[key.split("/", 1)[1]] = npz_data[key]
        else:
            metadata[key] = npz_data[key]
    return features, aux, metadata

cache_dir = Path(CACHE_DIR)
if not cache_dir.exists():
    print(f"Cache directory does not exist: {CACHE_DIR}")
else:
    npz_files = sorted(cache_dir.glob("*.npz"))
    print(f"Found {len(npz_files)} NPZ cache files")
    
    if npz_files:
        # Load the first cache file as an example
        first_npz = npz_files[0]
        print(f"\nInspecting: {first_npz.name}")
        print("=" * 80)
        
        with np.load(first_npz, allow_pickle=False) as npz_data:
            features, aux, metadata = _split_cache_arrays(npz_data)
            
            # Print features
            print("\nFEATURES:")
            print("-" * 80)
            if features:
                for key in sorted(features.keys()):
                    arr = features[key]
                    print(f"  {key:40s} shape={str(arr.shape):30s} dtype={arr.dtype}")
            else:
                print("  (no features)")
            
            # Print auxiliary data
            print("\nAUXILIARY DATA:")
            print("-" * 80)
            if aux:
                for key in sorted(aux.keys()):
                    arr = aux[key]
                    print(f"  {key:40s} shape={str(arr.shape):30s} dtype={arr.dtype}")
            else:
                print("  (no auxiliary data)")
            
            # Print metadata
            print("\nMETADATA:")
            print("-" * 80)
            if metadata:
                for key in sorted(metadata.keys()):
                    arr = metadata[key]
                    print(f"  {key:40s} shape={str(arr.shape):30s} dtype={arr.dtype}")
                    # Print scalar values for better readability
                    if arr.size == 1:
                        try:
                            if isinstance(arr.item(), bytes):
                                print(f"      value: {arr.item().decode('utf-8')}")
                            else:
                                print(f"      value: {arr.item()}")
                        except:
                            pass
            else:
                print("  (no metadata)")

Found 1000 NPZ cache files

Inspecting: training_tfexample.tfrecord-00000-of-01000.npz

FEATURES:
--------------------------------------------------------------------------------
  ego_state                                shape=(455, 5)                       dtype=float32
  ego_trajectory                           shape=(455, 25, 5)                   dtype=float32
  goal_xy                                  shape=(455, 2)                       dtype=float32
  map_features                             shape=(455, 128, 128, 25)            dtype=float32
  map_valid                                shape=(455, 128, 128)                dtype=bool
  other_states                             shape=(455, 128, 15)                 dtype=float32
  other_valid                              shape=(455, 128)                     dtype=bool
  remaining_timesteps                      shape=(455, 1)                       dtype=float32
  traffic_light_features                   shape=(455, 16, 9)              

In [2]:
# Optional: Load and inspect ALL cache files with statistics
print("\n" + "=" * 80)
print("SUMMARY ACROSS ALL CACHE FILES")
print("=" * 80)

all_feature_shapes: dict[str, list[tuple[int, ...]]] = {}
all_aux_shapes: dict[str, list[tuple[int, ...]]] = {}

for npz_file in npz_files:
    with np.load(npz_file, allow_pickle=False) as npz_data:
        features, aux, _ = _split_cache_arrays(npz_data)
        
        for key, arr in features.items():
            if key not in all_feature_shapes:
                all_feature_shapes[key] = []
            all_feature_shapes[key].append(arr.shape)
        
        for key, arr in aux.items():
            if key not in all_aux_shapes:
                all_aux_shapes[key] = []
            all_aux_shapes[key].append(arr.shape)

print("\nFEATURE SHAPES ACROSS FILES:")
print("-" * 80)
for key in sorted(all_feature_shapes.keys()):
    shapes = all_feature_shapes[key]
    print(f"  {key:40s}: {len(shapes)} files")
    for i, shape in enumerate(shapes[:3]):  # Show first 3
        print(f"      [{i}] {shape}")
    if len(shapes) > 3:
        print(f"      ... +{len(shapes) - 3} more")

print("\nAUXILIARY SHAPES ACROSS FILES:")
print("-" * 80)
for key in sorted(all_aux_shapes.keys()):
    shapes = all_aux_shapes[key]
    print(f"  {key:40s}: {len(shapes)} files")
    for i, shape in enumerate(shapes[:3]):  # Show first 3
        print(f"      [{i}] {shape}")
    if len(shapes) > 3:
        print(f"      ... +{len(shapes) - 3} more")


SUMMARY ACROSS ALL CACHE FILES


EOFError: No data left in file

In [ ]:
# Detailed inspection of a specific cache file
# You can change the index to inspect different files
FILE_INDEX = 0

if npz_files:
    target_npz = npz_files[FILE_INDEX]
    print(f"\nDETAILED INSPECTION OF: {target_npz.name}")
    print("=" * 80)
    
    with np.load(target_npz, allow_pickle=False) as npz_data:
        features, aux, metadata = _split_cache_arrays(npz_data)
        
        # Show feature details with sample values
        print("\nFEATURE DETAILS:")
        print("-" * 80)
        for key in sorted(features.keys()):
            arr = features[key]
            print(f"\n{key}:")
            print(f"  shape: {arr.shape}")
            print(f"  dtype: {arr.dtype}")
            print(f"  min: {arr.min():.4f}, max: {arr.max():.4f}, mean: {arr.mean():.4f}")
            
            # Show first sample
            if arr.size > 0:
                if arr.ndim == 1:
                    print(f"  sample: {arr[:min(5, len(arr))]}")
                elif arr.ndim == 2:
                    print(f"  first row: {arr[0, :min(10, arr.shape[1])]}")
                else:
                    print(f"  first element shape: {arr[0].shape}")
        
        # Show metadata details
        print("\n" + "-" * 80)
        print("METADATA:")
        print("-" * 80)
        for key in sorted(metadata.keys()):
            arr = metadata[key]
            print(f"{key}: shape={arr.shape}, dtype={arr.dtype}", end="")
            if arr.size == 1:
                try:
                    val = arr.item()
                    if isinstance(val, bytes):
                        print(f" value={val.decode('utf-8')}")
                    else:
                        print(f" value={val}")
                except:
                    print()
            else:
                print()